# 어텐션 메커니즘 구현하기 목차
* [Chapter 1 개요](#chapter1)
* [Chapter 2 긴 시퀸스 모델링의 문제점](#chapter2)
* [Chapter 3 어텐션 메커니즘으로 데이터 의존성 포착하기](#chapter3)
* [Chapter 4 셀프 어텐션으로 입력의 서로 다른 부분에 주의 기울이기](#chapter4)
    * [Section 4-1 훈련 가능한 가중치가 없는 간단한 셀프 어텐션 메커니즘](#section4-1)
    * [Section 4-2 모든 입력 토큰에 대해 어텐션 가중치 계산하기](#section4-2)
* [Chapter 5 훈련 가능한 가중치를 가진 셀프 어텐션 구현하기](#chapter5)
    * [Section 5-1 단계별로 어텐션 가중치 계산하기](#section5-1)

## Chapter 1 개요 <a class="anchor" id="chapter1"></a>
1. 어텐션 메커니즘을 살펴보고 내부가 어떻게 동작하는지 알아봅니다.

2. 셀프 어텐션 메커니즘을 둘러싼 다른 부분을 구현하여 작동 방식을 알아본다.

    ![구현단계](image/03-00-process2.png)

3. 네 가지 버전의 어텐션 메커니즘을 구현한다.
    - 이전에 구현한 것에 새로운 기능을 추가하는 식으로 만든다.

        ![구현단계](image/03-00-process3.png)




## Chapter 2 긴 시퀸스 모델링의 문제점 <a class="anchor" id="chapter2"></a>
1. LMM 시대 이전 어텐션 메커니즘을 사용하지 않던 구조가 가졌던 문제점
    - 한 언어에서 다른 언어로 텍스트를 번역하는 언어 모델을 만든다고 가정
    - 소스 언어와 타켓 언어가 문법 구조가 다르기 때문에 한 단어씩 번역할 수 없다.
    - 이 문제를 해결하기 위해 DNN에서 인코더와 디코더 2개의 서브모듈을 사용한다.
        - 인코더: 소스 언어 문장을 벡터로 인코딩
        - 디코더: 벡터를 타켓 언어 문장으로 디코딩

        ![번역](image/04-01-attention.png)    

    - 트랜스포머가 개발되기 전에는 순환 신경망(RNN recurrent neural network)을 사용했다.
        - 시퀸스 데이터를 처리하는 데 특화된 구조
        - 이전 시점의 출력을 현재 시점의 입력으로 사용하는 순환 구조를 가짐
        - 인코더가 입력 텍스트를 받아 순차적으로 처리
        - 인코더는 각 단계마다 은닉 상태를 업데이트 하며, 최종 은닉 상태로 입력 시퀸스 전체 의미를 포착
           - 인코더가 전체 입력 텍스트를 하나의 은닉 상태로 처리한다.
        - 디코더도 매 스텝마다 은닉 상태를 업데이트하며 이를 통해 다음 단어 예측에 필요한 문맥정보를 다음 스텝에 전달
           - 디코더가 은닉 상태를 받아 출력을 생성한다.
        - 은닉 상태를 임베딩 벡터 개념으로 생각할 수 있다.
        - 디코딩 단계에서 RNN이 이전 은닉 상태를 참조할 수 없다.
        - RNN은 긴 시퀸스를 처리하는 데 어려움이 있다.
           - 최종 은닉 상태에만 의존하게 되어 맥락을 놓칠 수 있다.
           - 멀리 떨어진 단어에 의존성이 있는 복잡한 문장의 경우는 특히 그렇다.

            ![RNN](image/04-01-RNN.png)    

## Chapter 3 어텐션 메커니즘으로 데이터 의존성 포착하기 <a class="anchor" id="chapter3"></a>
1. RNN이 인코딩된 전체 입력을 하나의 은닉 상태에 저장해서 디코더에 전달하는 문제를 해결하기 위해 바흐다나우 어텐션 메커니즘이 개발되었다.

2. 바흐다나우 어텐션
    - 인코더-디코더 RNN을 수정하여 디코딩 단계마다 디코더가 선택적으로 입력 시퀸스의 서로 다른 부분을 참조할 수 있다.
    - 디코더가 매 스텝마다 인코더의 모든 은닉 상태를 참조할 수 있다.
    - 3년 후에 RNN 구조가 자연어 처리를 위한 심층 신경망을 구축하는데 필수적이지 않다고 밝혀졌다.

        ![RNN](image/03-02-battention.png)    


3. 트랜스포머 모델이 등장했고, 바흐나나우 어텐션에 영감을 받은 셀프 어텐션이 포함되었다.
    - 셀프 어텐션은 입력 시퀸스의 서로 다른 위치 간의 의존성을 포착하는 메커니즘
    - 트랜스포머 모델은 RNN을 사용하지 않고도 긴 시퀸스를 효과적으로 처리할 수 있다.
    - 트랜스포머 모델은 자연어 처리 분야에서 혁신을 일으켰고, 이후 대형 언어 모델(LLM)의 발전에 중요한 역할을 했다.

4. 셀프 어텐션은 트랜스포머에서 사용되는 메커니즘
    - 입력 시퀸스에 있는 각 위치가 동일 시퀸스에 있는 다른 모든 위치와 상호작용하여 중요도를 부여한다.

## Chapter 4 셀프 어텐션으로 입력의 서로 다른 부분에 주의 기울이기 <a class="anchor" id="chapter4"></a>
1. 셀프 어텐션은 트랜스포머 구조를 바탕으로 하는 모든 LLM의 기반이 된다.
    - 셀프 어텐션의 '셀프'는 하나의 입력 시퀸스에 있는 서로 다른 위치의 원소 사이에서 어텐션 가중치를 계산한다.
    - 문장 안의 단어와 이미지 안에 있는 픽셀 사이의 관계와 의존성을 평가하고 학습한다.
    - 2 개의 다른 시퀸스에 있는 원소 사이의 관계에 초점을 맞추는 전통적인 어텐션 메커니즘과는 다르다.



### Section 4-1 훈련 가능한 가중치가 없는 간단한 셀프 어텐션 메커니즘 <a class="anchor" id="section4-1"></a>
1. 목표는 훈련 가능한 가중치를 추가하기 전에 셀프 어텐션에 있는 몇 가지 핵심 개념을 이해하는 것이다.
    - 셀프 어텐션의 목표는 다른 모든 입력 원소의 정보를 조합하여 각각의 입력 원소에 대한 문맥 벡터를 계산하는 것이다.

2. "Your journey starts with one step"이라는 입력 텍스트
    - x<sup>(1)</sup>에 해당하는 시퀴스의 각 원소는 "your"를 표현하는 d 차원 임베딩 벡터이다. 
    - 셀프 어텐션에서는 입력 시퀴스에 있는 각 원소 x<sup>(i)</sup>에 대한 문캑 벡터 z<sup>(i)</sup>를 계산하는 것이 목표이다.
       - 문맥 벡터는 정보가 풍부한 임베딩 벡터로 생각할 수 있다.
    - 두 번재 입력 원소 "journey" x<sup>(2)</sup>에 대한 임베딩 벡터와 문맥 벡터 z<sup>(2)</sup>를 살펴보면.
       - 문맥 벡터 z<sup>(2)</sup>는  x<sup>(2)</sup>와 다른 모든 입력 (x<sup>(1)</sup>~x<sup>(i)</sup>) 사이의 정보를 담은 임베딩이다.

3. 문맥 벡터는 셀프 어텐션에서 매우 중요한 역활을 한다.
    - 문맥 벡터의 목적은 입력 시퀸스에 있는 다른 모든 원소의 정보를 통합해 이 시퀸스에 있는 각 원소의 표현을 풍부하게 만드는 것이다.
    - LLM에서 문장에서 다른 단어 사이의 관계와 관련성을 이해하는 것이 필요하다

        ![셀프 어텐션](image/03-03-attenction.png)  

4. 셀프 어텐션을 구하는 첫 번째 단계는 어텐션 점수(attention score)를 계산하는 것이다.
    - 어텐션 점수는 입력 시퀸스에 있는 서로 다른 원소 사이의 관련성을 측정한다.
    - 어텐션 점수는 두 입력 원소 x<sup>(i)</sup>와 x<sup>(j)</sup> 사이의 유사성을 나타낸다.
    - 두 번째 입력 원소  x<sup>2</sup>를 쿼리로 사용하여 문맥 벡터 z<sup>(2)</sup>를 계산한다
    - 쿼리 x<sup>(2)</sup>와 다른 모든 원소 사이의 점곱(dot product)을 계산하여 어텐션 점수를 w를 구한다.
    - 점곱은 두 벡터 사이의 유사성을 측정하는 간단한 방법이다.
       - 점곱이 클수록 두 벡터가 얼마나 가까이 놓여 있는지 정량화 할 수 있다.
    - 셀프 어텐션에서 점곱은 시퀸스에 있는 각 원소가 다른 소에 얼마나 관련이 있는지 평가하는 데 사용된다.

        ![셀프 어텐션 점수](image/03-03-score.png)  

In [4]:
import torch

# 입력 시퀸스 (6개의 단어, 각 단어는 3차원 벡터로 표현 - 단어 임베딩)
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

# 쿼리 토큰과 각 입력 토큰 사이의 어텐션 점수를 점 곱으로 계산
query = inputs[1] # 두 번째 입력 토큰을 쿼리 토큰으로 사용 (journey)
atten_scores_2 = torch.empty(inputs.shape[0]) # 어텐션 점수를 저장할 텐서

# 각 입력 토큰에 대해 쿼리 토큰과의 점 곱 계산
for i, x_i in enumerate(inputs):
    atten_scores_2[i] = torch.dot(x_i, query)
    
print(atten_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


5. 다음 단계로 어텐션 점수를 정규화한다.
    - 정규화를 하는 목적은 어텐션 가중치의 합이 1이 되도록 하는 것이다.
    - 정규화를 하면 해석이 용이하고 LMM을 훈련할 때 안정성을 유지하는데 도움이 된다.

6. 입력 쿼리 x<sup>(2)</sup>에 대한 어텐션 가중치 ω<sub>21</sub>에서  ω<sub>2T</sub>까지 구한다.
    - 다음 단계는 어텐션 점수를 정규화하여 어텐션 가중치 α<sub>21</sub>에서  α<sub>2T</sub>까지 구하는 것이다.

        ![가중치](image/03-03-weight.png)

In [5]:
atten_weights_2_tmp = atten_scores_2 / atten_scores_2.sum()  # 어텐션 점수를 정규화하여 어텐션 가중치 계산
print("어텐션 가중치: ", atten_weights_2_tmp)
print("합: ", atten_weights_2_tmp.sum())

어텐션 가중치:  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
합:  tensor(1.0000)


7. 일반적으로 소프트맥스 함수를 사용하여 정규화를 진행한다.
   - 어텐션 가중치가 항상 양수가 되도록 보장한다.
   - 가중치가 높을 수록 중요도가 높다.
   - 큰 입력아니 매우 작은 입력을 처리 할 때 오버플로나 언더플로 같은 수치 불안정 문제 발생할 수 있다.
   - 실전에 광법위하게 성능 최적화가 된 파이토치의 소프트맥스 함수를 사용한다.

In [6]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum()

atten_weights_2_naive = softmax_naive(atten_scores_2)
print("어텐션 가중치: ", atten_weights_2_naive)
print("합: ", atten_weights_2_naive.sum())

어텐션 가중치:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
합:  tensor(1.)


In [7]:
atten_weights_2 = torch.softmax(atten_scores_2, dim=0)
print("어텐션 가중치: ", atten_weights_2)
print("합: ", atten_weights_2.sum())

어텐션 가중치:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
합:  tensor(1.)


8. 임베딩된 입력 토큰 x<sup>(i)</sup>와 각 토큰에 해당하는 어턴션 가중치를 곱한 후 모두 더해서 문맥 벡터 z<sup>(i)</sup>를 계산한다.
    - 문맥 벡터 z<sup>(i)</sup>는 입력 쿼리 x<sup>(i)</sup>와 시퀸스에 있는 다른 모든 원소 사이의 정보를 통합한 것이다.
    - 문맥 벡터 z<sup>(i)</sup>는 입력 쿼리 x<sup>(i)</sup>와 시퀸스에 있는 다른 모든 원소 사이의 관련성을 반영한다.

        ![문맥 벡터](image/03-03-context2.png)

In [8]:
query = inputs[1] # 두 번째 입력 토큰을 쿼리 토큰으로 사용 (journey)
context_vec2 = torch.zeros(inputs.shape[1]) # 문맥 벡터를 저장할 텐서
for i, x_i in enumerate(inputs):
    context_vec2 += atten_weights_2[i] * x_i
print("문맥 벡터: ", context_vec2)

문맥 벡터:  tensor([0.4419, 0.6515, 0.5683])


### Section 4-2 모든 입력 토큰에 대해 어텐션 가중치 계산하기 <a class="anchor" id="section4-2"></a>
1. 지금까지 노란 색으로 강조된 부분에 대한 어텐션 가중치와 문맥 벡터를 계산했다.

    ![어텐션 가중치](image/03-03-weight2.png)

2. 두 번째 원소 z<sub>2</sub>뿐만 아니라 모든 문맥 벡터를 계산한다.

In [ ]:
atten_scores = torch.empty(6, 6) # 어텐션 점수를 저장할 텐서

# 모든 입력 토큰에 대해 쿼리 토큰과의 점 곱 계산
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        atten_scores[i, j] = torch.dot(x_i, x_j)
print(atten_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
# for 루프는 느리기 때문에 행렬 곱셈을 사용한다.
attn_scores = inputs @ inputs.T # 행렬 곱셈
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
# 정규화 진행
attn_weights = torch.softmax(attn_scores, dim=1) # 1: 마지막 차원을 기준으로 정규화 진행
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [14]:
# 각 행의 값을 더해 1이 되는지 확인
row_2_sum = attn_weights[2].sum()
print("세 번째 행의 합: ", row_2_sum)

# 모든 행의 합이 1인지 확인
all_rows_sum = attn_weights.sum(dim=-1)
print("모든 행의 합: ", all_rows_sum)

세 번째 행의 합:  tensor(1.0000)
모든 행의 합:  tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [15]:
# 어텐션 가중치와 입력 행렬을 곱해서 문맥 벡터 계산하기
all_context_vecs = attn_weights @ inputs
print("모든 문맥 벡터:\n", all_context_vecs)

모든 문맥 벡터:
 tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [16]:
# 두 번째 문맥 벡터와 비교
print("두 번째 문맥 벡터:\n", context_vec2)

두 번째 문맥 벡터:
 tensor([0.4419, 0.6515, 0.5683])


## Chapter 5 훈련 가능한 가중치를 가진 셀프 어텐션 구현하기 <a class="anchor" id="chapter5"></a>
1. 훈련 가능한 가중치를 가진 셀프 어텐션 메커니즘은 스케일드 점곱 어텐션이라고 부른다.
    - 특정 입력 원소에 대한 입력 벡터의 가중치 합으로 문맥 벡터를 계산한다.
    - 모델 훈련 과정에서 업데이트되는 훈련 가능한 가중치 행렬이 추가된다.
        - 가중치 행렬을 통해 모데이 '좋은' 문맥 벡터를 생성하는 방법을 학습한다. 

### Section 5-1 단계별로 어텐션 가중치 계산하기 <a class="anchor" id="section5-1"></a>
1. 훈련 가능한 가중치 행렬 3개, 즉 W<sub>q</sub>, W<sub>k</sub>, W<sub>v</sub>를 추가하여 셀프 어텐션 메커니즘을 단계별로 구현한다.
    - 3개의 행렬을 사용해 임베딩된 입력 토큰 x<sup>(i)</sup>를 각각 쿼리(q<sup>(i)</sup>), 키(k<sup>(i)</sup>), 값(v<sup>(i)</sup>) 벡터로 투영한다.
    - 입력 원소 x에 대한 쿼리(q), 키(k), 값(v) 벡터 계산
    - 두 번째 입력 원소 x<sup>(2)</sup>를 쿼리의 입력으로 사용 
    - 쿼리 벡터는 입력과 행렬 W를 곱해서 계산한다.

        ![단계별 가중치](image/03-04-stepWeight.png)

2. 문맥 벡터 z<sup>(2)</sup>를 구한 후 코드를 수정하여 모든 문맥 벡터를 구한다.

    ![테이블](image/03-04-table3.png)

In [ ]:
x_2 = inputs[1] # 두 번째 입력 토큰 (journey)
d_in = inputs.shape[1] # 입력 임베딩의 크기, 3
d_out = 2 # 출력 임베딩의 크기, 2

# 3개의 가중치 행렬 Wq, Wk, Wv 초기화
# requires_grad=False로 설정하여 훈련 과정에서 업데이트되지 않도록 함
#   - 모델 훈련 시 requires_grad=True로 변경해야 함
W_query = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False) # 쿼리 가중치 행렬
W_key = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False) # 키 가중치 행렬
W_value = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False) # 값 가중치 행렬

print("쿼리 가중치 행렬:\n", W_query)
print("키 가중치 행렬:\n", W_key)
print("값 가중치 행렬:\n", W_value)

쿼리 가중치 행렬:
 Parameter containing:
tensor([[-2.0832,  2.4916],
        [ 0.3060, -1.9368],
        [ 0.3950,  1.4558]])
키 가중치 행렬:
 Parameter containing:
tensor([[ 0.3260,  1.8413],
        [-1.0984,  1.8216],
        [-1.4147,  1.1726]])
값 가중치 행렬:
 Parameter containing:
tensor([[ 0.7324, -1.5040],
        [-0.4526,  0.7108],
        [-0.5131,  1.0748]])


In [32]:
# 쿼리, 키, 값 벡터 계산
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

# 쿼리의 출력 결과는 2차원 벡터
print("입력 벡터: ", x_2)
print("쿼리 벡터: ", query_2)
print("키 벡터: ", key_2)
print("값 벡터: ", value_2)

입력 벡터:  tensor([0.5500, 0.8700, 0.6600])
쿼리 벡터:  tensor([-0.6188,  0.6462])
키 벡터:  tensor([-1.7101,  3.3714])
값 벡터:  tensor([-0.3296,  0.5006])


In [ ]:
# 키, 값을 편하게 가져오도록 미리 계산
keys = inputs @ W_key   # 모든 입력 토큰에 대한 키 벡터 계산
values = inputs @ W_value # 모든 입력 토큰에 대한 값 벡터 계산
print("모든 키 벡터:\n", keys, keys.shape)
print("모든 값 벡터:\n", values, values.shape)

모든 키 벡터:
 tensor([[-1.2837,  2.1086],
        [-1.7101,  3.3714],
        [-1.6533,  3.3484],
        [-1.0322,  1.8486],
        [-0.1650,  1.9905],
        [-1.6406,  2.1943]]) torch.Size([6, 2])
모든 값 벡터:
 tensor([[-0.2096,  0.4165],
        [-0.3296,  0.5006],
        [-0.2956,  0.4348],
        [-0.2707,  0.4361],
        [ 0.3995, -0.8729],
        [-0.6077,  1.0846]]) torch.Size([6, 2])


3. 각각의 가중치 행렬로 입력을 변환한 쿼리와 키 벡터 사이의 점곱을 계산하여 어텐션 점수를 구한다.
    - 두 번째 입력원소("journey")의 쿼리 벡터와 모든 입력 원소의 키 벡터 사이의 점곱을 계산하여 어텐션 점수를 구한다.

    ![점곱 계산](image/03-04-step2.png)

In [34]:
# 어텐션 점수 W22 계산
keys_2 = keys[1] # 두 번째 입력 토큰에 대한 키 벡터
print("두 번째 키 벡터: ", keys_2, keys_2.shape)

attn_scores_22 = query_2.dot(keys_2)
print("어텐션 점수 W22: ", attn_scores_22)


두 번째 키 벡터:  tensor([-1.7101,  3.3714]) torch.Size([2])
어텐션 점수 W22:  tensor(3.2369)


In [ ]:
# 행렬 곱셈으로 일반화하여 모든 어텐션 점수 계산
atten_score_2 = query_2 @ keys.T

# 두 번째 원소가 앞서 계산한 attn_scores_22와 동일하다. 3.2369
print("모든 어텐션 점수:\n", atten_score_2)


모든 어텐션 점수:
 tensor([2.1569, 3.2369, 3.1868, 1.8333, 1.3884, 2.4332])


4. 어텐션 점수에서 어텐션 가중치를 구한다.
    - 소프트맥스 함수를 사용하여 어텐션 점수를 정규화한다.
    - 어텐션 점수를 키의 임베딩 차원의 제곱근으로 나눈다.
       - 제곱근은 0.5를 제곱하는 것과 수학적으로 동일하다.
    - 임베딩 차원 크기로 정규화를 하는 이유는 그레이디언트가 작아지는 것을 피하여 성능을 향상시키기 위해서이다.
    - 임베딩 차원이 커지면 소프트맥스 함수 때문에 역전파 과정에서 매우 작은 그레이디언트를 생성할 수 있다.
    - 임베딩 차원을 제곱근으로 나누기때문에 셀프 어텐션 메커니즘을 스케일드 점곱 어텐션이라고 부른다.

        ![어텐션 가중치](image/03-04-step3.png)

In [36]:
d_k = keys.shape[-1] # 키 벡터의 차원
attn_weights_2 = torch.softmax(atten_score_2 / (d_k ** 0.5), dim=-1)
print("어텐션 가중치: ", attn_weights_2)
print("합: ", attn_weights_2.sum())

어텐션 가중치:  tensor([0.1281, 0.2748, 0.2652, 0.1019, 0.0744, 0.1557])
합:  tensor(1.)


5. 문맥 벡터를 계산한다.
    - 모든 값 벡터를 어텐션 가중치를 통해 결합하여 문맥 벡터를 계산한다.

        ![문맥 계산](image/03-04-step4.png)

In [37]:
context_vec_2 = attn_weights_2 @ values
print("문맥 벡터: ", context_vec_2)

문맥 벡터:  tensor([-0.2883,  0.4546])
